# Cross-Silo FTL — full reproducible run

## 0. Environment (set before torch is imported)

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not on Colab or already mounted:", e)

# ---- what to run ----
RUN_ABLATION = True     # set False to skip the (slow) component ablation
RUN_TSNE     = True

## 1. Paths

In [ ]:

PROJECT = "/content/drive/MyDrive/Depression_FL_Project"

# Stage 0 output
LABELS_CSV = f"{PROJECT}/fl/labels_v1/fl_harmonized_labels.csv"

EEG_EMBED_DIR = f"{PROJECT}/data/embeddings"   # eeg_embeddings_{train,val,test}.npy + eeg_subj_ids_* + eeg_labels_*

TEXT_EMBED_NPZ = f"{PROJECT}/output/PDCH/processed_v1/embeddings/pdch_text_emb_full_dialogue.npz"

AUDIO_TURN_DIR = f"{PROJECT}/audio/v1/turn_embeddings"
AUDIO_LOCK_LAYER = 10   # matches the locked model's chosen WavLM layer

OUT = f"{PROJECT}/fl/fusion_v1"
import os, glob, json
import numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)

for p in (LABELS_CSV, EEG_EMBED_DIR, TEXT_EMBED_NPZ, AUDIO_TURN_DIR):
    print(f"  exists={os.path.exists(p)}  {p}")

## 2. Imports

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd
import math
import matplotlib.pyplot as plt
import random, functools, copy
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from scipy import stats
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3. Determinism

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)                 # <-- the RNG that mixup uses; was never seeded
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def enable_determinism():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print("[determinism] use_deterministic_algorithms unavailable:", e)

enable_determinism()
print("[determinism] cuDNN deterministic ON, benchmark OFF")

## 4. Core pipeline (embedded)

In [ ]:
CFG_SRC = '# ============================ FL CONFIG ============================\n\n# --------------------------------------------------\n# Classification task: binary depressed/not  vs  3-class severity\n# --------------------------------------------------\n# Binary (depressed vs not) is the clinically standard depression-detection target, and the\n# frozen embeddings carry far more binary signal than 3-class severity (e.g. audio layer-10\n# scores ~0.56 macro-F1 binary vs ~0.34 on 3-class). It is the realistic route to >0.5 macro-F1\n# across ALL three silos. \'severity3\' restores the original 3-class task. Binary collapses the\n# harmonized label y_unified: class 0 = not depressed, classes >= BINARY_POSITIVE_FROM = depressed.\nTASK = _TASK   # injected by run_setup()           # "binary" | "severity3"\n# Per-silo positive cut on y_unified (>= value -> positive). EEG and AUDIO keep the standard\n# depressed-vs-not cut (>=1) and are left UNCHANGED (they already score well). TEXT (PDCH) is a\n# depression-clinic cohort, ~[8,51] at >=1, which caps macro-F1; the >=2 cut (\'severe vs\n# non-severe\') rebalances it to ~[38,21] and gives real headroom. Use an int to apply one cut\n# to all silos, or this dict for a per-silo cut.\n# WRITE-UP CAVEAT: text then answers \'severe vs non-severe\' while eeg/audio answer \'depressed\n# vs not\' -- state this explicitly in the methods section.\nBINARY_POSITIVE_FROM = {"eeg": 1, "audio": 1, "text": 2}\nN_CLASSES    = 2 if TASK == "binary" else 3\n\n# --------------------------------------------------\n# Decision-threshold tuning (binary only) -- the main lever for the imbalanced TEXT silo\n# --------------------------------------------------\n# PDCH is a depression-clinic dataset: at the depressed-vs-not cut its train split is ~[8, 51],\n# so an argmax (0.5) classifier just predicts \'depressed\' for everyone and macro-F1 stalls ~0.47.\n# We instead pick EACH silo\'s decision threshold on ITS OWN validation set (maximizing val\n# macro-F1) and apply it to test. Tuned on val, applied to test -> no leakage. Balanced silos\n# land near 0.5 (no change); the imbalanced text silo benefits most.\nUSE_THRESHOLD_TUNING = True\nTHRESHOLD_GRID       = 41    # candidate thresholds spanning (0.05 .. 0.95)\nSHARED_DIM   = 512\n\n# More local optimization per round\nLOCAL_EPOCHS = 10\n\nN_ROUNDS     = 100\nEARLY_STOP_PATIENCE = 20\nWARMUP_ROUNDS = 15\n\nLR_HEAD      = 1e-4\n\nN_CLIENTS    = 3\n\n# Projection head capacity -- REDUCED for the tiny-n regime (was 768x3, which badly\n# over-parameterized ~33-113 training subjects). Smaller head + LayerNorm generalizes better.\nHIDDEN_DIM   = 256\nN_HEAD_LAYERS = 2\nHEAD_DROPOUT = 0.2\n\n# --------------------------------------------------\n# Performance levers added for this run\n# --------------------------------------------------\nUSE_INPUT_STANDARDIZATION = True   # z-score head inputs per silo (TRAIN stats only)\nBATCH_SIZE                = 32      # mini-batch local training (was full-batch)\nAUGMENT_NOISE_STD         = 0.05    # Gaussian embedding jitter -> cheap augmentation\nUSE_CLASS_WEIGHTING       = True    # inverse-freq weighting in the prototype loss (macro-F1)\nPROTO_TEMPERATURE         = 0.1     # prototype contrastive temperature\n\nGRAD_CLIP    = 1.0\n\n# --------------------------------------------------\n# Aggregation\n# --------------------------------------------------\nUSE_SAMPLE_SIZE_WEIGHTING = False\nUSE_CLIENT_RELIABILITY_WEIGHTING = True\n\n# --------------------------------------------------\n# Losses\n# --------------------------------------------------\nUSE_COMPACTNESS = True\nCOMPACT_LAMBDA  = 0.20\n\n# --------------------------------------------------\n# Learned per-silo classifier head (FedPer-style)  <-- main lever for lifting audio/text\n# --------------------------------------------------\n# The original pipeline predicts by nearest-class-mean (cosine to the global prototype),\n# which is a weaker decision rule than even a logistic-regression probe -- that is why FL\n# underperformed the standalone audio probe. We add a small trained classifier on top of the\n# (FL-aligned) projected representation, trained with class-balanced / focal loss + optional\n# mixup, which directly targets minority-class recall (i.e. macro-F1). The prototype loss is\n# kept as the CROSS-SILO ALIGNMENT regularizer -- that is still the transfer mechanism.\nUSE_CLASSIFIER_HEAD = True       # add a trained classifier on the projected rep\nPREDICT_WITH        = \'classifier\'   # \'classifier\' | \'prototype\'  (eval + early-stop metric)\nCE_WEIGHT           = 1.0        # weight of the classification loss in the local objective\nUSE_FOCAL_LOSS      = True       # focal vs plain (class-weighted) cross-entropy\nFOCAL_GAMMA         = 1.5        # focal focusing parameter (0 -> plain weighted CE)\nCE_LABEL_SMOOTHING  = 0.05       # label smoothing for plain CE (ignored under focal)\nUSE_MIXUP           = True       # mixup augmentation on the classification path only\nMIXUP_ALPHA         = 0.2        # Beta(alpha,alpha) mixing strength\n\nN_SEEDS = 5\n\n# --------------------------------------------------\n# Prototype Stability\n# --------------------------------------------------\nUSE_ETF_INIT = True\n\nPROTO_EMA_MOMENTUM = 0.95\nGATE_ADAPT_MOMENTUM = 0.95\n\nGATE_MIN = 0.05\nGATE_MAX = 0.95\n\n# --------------------------------------------------\n# Modality Personalization\n# --------------------------------------------------\n# Audio should stay mostly local\n# Text moderately local\n# EEG gets most global guidance\n\nGATE_INIT = {\n    # audio recalibrated: simplification removed a redundant 0.05x multiplier that was stacked\n    # on top of this gate, which silently 20x\'d audio\'s effective global-prototype pull and\n    # reversed its gain (0.369 -> 0.332, back below standalone 0.343). This value bakes the\n    # SAME empirically-known-good effective strength into the one remaining mechanism directly,\n    # instead of reintroducing a second hardcoded layer.\n    "audio": 0.0075,\n    "eeg":   0.75,\n    "text":  0.35,\n}\n\nWEIGHT_CLIP_RANGE = (0.3, 1.5)\n\nLR_CANDIDATES = [\n    5e-4,\n    2e-4,\n    1e-4,\n    5e-5,\n    2e-5\n]\n\n# --------------------------------------------------\n# Auto-load previously-searched best config as the standing baseline\n# --------------------------------------------------\n# After the hyperparameter-search cell runs once, it writes fl_hparam_search_best.json.\n# On subsequent runs this block reloads those values so the tuned config becomes the\n# baseline automatically. Set FL_USE_SEARCHED_BASELINE=False to ignore it and use the\n# hand-set defaults above instead.\nFL_USE_SEARCHED_BASELINE = True\ntry:\n    _best_path = f"{OUT}/fl_hparam_search_best.json"\n    if FL_USE_SEARCHED_BASELINE and os.path.exists(_best_path):\n        _b = json.load(open(_best_path)); _bc = _b.get("best", {})\n        for _k in ("LR_HEAD", "COMPACT_LAMBDA", "PROTO_TEMPERATURE", "AUGMENT_NOISE_STD",\n                   "HEAD_DROPOUT", "HIDDEN_DIM", "LOCAL_EPOCHS",\n                   "CE_WEIGHT", "FOCAL_GAMMA"):\n            if _k in _bc:\n                globals()[_k] = int(_bc[_k]) if _k in ("HIDDEN_DIM", "LOCAL_EPOCHS") else _bc[_k]\n        if "GATE_INIT" in _b:\n            GATE_INIT = {k: float(v) for k, v in _b["GATE_INIT"].items()}\n        print(f"[baseline] loaded searched config from {_best_path} "\n              f"(val macro-F1={_bc.get(\'val_macro_f1\',\'?\')}) -> overriding the defaults above")\n    else:\n        print("[baseline] using hand-set defaults (no searched config found yet)")\nexcept Exception as _e:\n    print(f"[baseline] could not load searched config ({_e}); using hand-set defaults")\n\nprint(f"N_CLIENTS={N_CLIENTS}")\nprint(f"LOCAL_EPOCHS={LOCAL_EPOCHS}")\nprint(f"N_ROUNDS={N_ROUNDS}")\nprint(f"WARMUP_ROUNDS={WARMUP_ROUNDS}")\nprint(f"HIDDEN_DIM={HIDDEN_DIM}")\nprint(f"N_HEAD_LAYERS={N_HEAD_LAYERS}")\nprint(f"USE_CLIENT_RELIABILITY_WEIGHTING={USE_CLIENT_RELIABILITY_WEIGHTING}")\nprint(f"GATE_INIT={GATE_INIT}")\n\n'

CLASSES_SRC = 'import torch, torch.nn as nn, torch.nn.functional as F\nimport numpy as np, pandas as pd\nimport math\nimport matplotlib.pyplot as plt\n\n# ============================ Core FL classes/functions ============================\n# (mirrors fl_fusion_core.py, unit-tested separately on synthetic heterogeneous-dim data)\n\ndef _safe_normalize(x, dim=-1, eps=1e-12):\n    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))\n\ndef make_simplex_etf(n_classes, dim, device=None, dtype=torch.float32):\n    """\n    Deterministic ETF / simplex initialization.\n    For 3 classes this gives an equiangular triangle embedded in SHARED_DIM.\n    """\n    if n_classes < 2:\n        raise ValueError("n_classes must be >= 2")\n    if n_classes == 2:\n        base = torch.tensor([[1.0], [-1.0]], dtype=dtype, device=device)\n    elif n_classes == 3:\n        base = torch.tensor([\n            [1.0, 0.0],\n            [-0.5,  math.sqrt(3) / 2.0],\n            [-0.5, -math.sqrt(3) / 2.0],\n        ], dtype=dtype, device=device)\n    else:\n        eye = torch.eye(n_classes, dtype=dtype, device=device)\n        eye = eye - eye.mean(dim=0, keepdim=True)\n        q, _ = torch.linalg.qr(eye.T, mode="reduced")\n        base = q.T[:n_classes]\n    if base.shape[1] < dim:\n        pad = torch.zeros((n_classes, dim - base.shape[1]), dtype=dtype, device=device)\n        base = torch.cat([base, pad], dim=1)\n    elif base.shape[1] > dim:\n        base = base[:, :dim]\n    return _safe_normalize(base, dim=-1)\n\ndef _init_proto_bank(n_classes=N_CLASSES, dim=SHARED_DIM, use_etf=USE_ETF_INIT, device=None):\n    if use_etf:\n        return {c: make_simplex_etf(n_classes, dim, device=device)[c].detach().clone()\n                for c in range(n_classes)}\n    g = torch.randn(n_classes, dim, device=device)\n    g = _safe_normalize(g, dim=-1)\n    return {c: g[c].detach().clone() for c in range(n_classes)}\n\nclass ProjectionHead(nn.Module):\n    """Native embedding dim -> shared 128-d. Configurable depth/width (HIDDEN_DIM, N_HEAD_LAYERS).\n    Outputs are L2-normalized so prototypes live on the unit hypersphere."""\n    def __init__(self, in_dim, out_dim=None, hidden_dim=None, n_layers=None, dropout=None):\n        super().__init__()\n        # Resolve from live globals at call time so a hyperparameter search can vary these\n        # by setting the module-level globals between trials (default args bind once at def\n        # time and would go stale inside a search loop).\n        out_dim    = SHARED_DIM    if out_dim    is None else out_dim\n        hidden_dim = HIDDEN_DIM    if hidden_dim is None else hidden_dim\n        n_layers   = N_HEAD_LAYERS if n_layers   is None else n_layers\n        dropout    = HEAD_DROPOUT  if dropout    is None else dropout\n        layers = []\n        d = in_dim\n        for _ in range(n_layers):\n            # LayerNorm (not BatchNorm): sample-independent, stable at tiny batch sizes / batch=1\n            layers += [nn.Linear(d, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout)]\n            d = hidden_dim\n        layers += [nn.Linear(d, out_dim)]\n        self.net = nn.Sequential(*layers)\n        self.identity_path = (in_dim == out_dim)\n        # FedPer-style local classifier on the projected rep (trained jointly; its params are\n        # already covered by the per-silo optimizer since it is a submodule of this head).\n        if USE_CLASSIFIER_HEAD:\n            self.classifier = nn.Sequential(\n                nn.Linear(out_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),\n                nn.Dropout(dropout), nn.Linear(hidden_dim, N_CLASSES))\n        else:\n            self.classifier = None\n\n    def forward(self, x):\n        out = self.net(x)\n        if self.identity_path:\n            out = 0.5 * x + 0.5 * out\n        return F.normalize(out, dim=-1)\n\n    def classify(self, emb):\n        """Logits from the projected (already L2-normalized) representation."""\n        return self.classifier(emb)\n\ndef compute_sample_size_weights(client_data, clip_range=WEIGHT_CLIP_RANGE):\n    """Kept for ablation only. Main runs use adaptive client gates + class-wise reliability."""\n    sizes = {silo: len(y) for silo, (X, y) in client_data.items()}\n    total = sum(sizes.values())\n    n_clients = len(sizes)\n    raw = {silo: (total / (n_clients * n)) for silo, n in sizes.items()}\n    lo, hi = clip_range\n    return {silo: float(np.clip(w, lo, hi)) for silo, w in raw.items()}\n\ndef compute_local_prototypes(emb, y, n_classes=N_CLASSES):\n    protos = {}\n    for c in range(n_classes):\n        mask = (y == c)\n        n = int(mask.sum().item())\n        if n > 0:\n            protos[c] = (F.normalize(emb[mask].mean(dim=0).detach(), dim=-1), n)\n    return protos\n\ndef aggregate_prototypes_naive(client_protos, n_classes=N_CLASSES):\n    """Vanilla FedProto: weighted-average by sample count."""\n    g = {}\n    for c in range(n_classes):\n        vecs, w = [], []\n        for cp in client_protos:\n            if c in cp:\n                v, n = cp[c]\n                vecs.append(F.normalize(v, dim=-1))\n                w.append(n)\n        if vecs:\n            w = torch.tensor(w, dtype=torch.float32)\n            w = w / w.sum().clamp_min(1e-12)\n            g[c] = _safe_normalize((torch.stack(vecs) * w.unsqueeze(-1)).sum(dim=0), dim=-1)\n    return g\n\ndef aggregate_prototypes_reliability(\n    client_protos,\n    client_reliability,\n    n_classes=N_CLASSES,\n    eps=1e-8\n):\n    """\n    Reliability-weighted prototype aggregation.\n\n    SIMPLIFIED (per literature-review report discussion): removed the hardcoded modality\n    multipliers (audio 1.50x / text 1.25x / eeg 1.00x) that were stacked on top of the\n    per-class reliability score. That was a second, redundant personalization layer on top\n    of the adaptive client_gates mechanism already in run_one_round -- two overlapping\n    hand-tuned mechanisms pulling the same direction makes the result hard to attribute and\n    introduces unexplained constants a thesis reviewer would reasonably question. The adaptive\n    gate (GATE_INIT + momentum adaptation) is now the SINGLE mechanism driving per-silo\n    personalization; this function relies purely on the per-class reliability signal itself.\n    """\n\n    g = {}\n\n    silo_names = list(client_reliability.keys())\n\n    for c in range(n_classes):\n\n        vecs = []\n        weights = []\n\n        for idx, cp in enumerate(client_protos):\n\n            if c not in cp:\n                continue\n\n            proto, _ = cp[c]\n\n            if idx < len(silo_names):\n                silo_name = silo_names[idx]\n\n                rel = float(\n                    client_reliability\n                    .get(silo_name, {})\n                    .get(c, 0.0)\n                )\n            else:\n                rel = 0.0\n\n            rel = max(rel, eps)\n\n            vecs.append(\n                F.normalize(proto, dim=-1)\n            )\n\n            weights.append(rel)\n\n        if len(vecs) == 0:\n            continue\n\n        weights = torch.tensor(\n            weights,\n            dtype=torch.float32,\n            device=vecs[0].device\n        )\n\n        weights = (\n            weights /\n            weights.sum().clamp_min(eps)\n        )\n\n        g[c] = _safe_normalize(\n            (\n                torch.stack(vecs)\n                * weights.unsqueeze(-1)\n            ).sum(dim=0),\n            dim=-1\n        )\n\n    return g\n\nclass TrainableGlobalPrototypes(nn.Module):\n    """FedTGP-style trainable global prototypes, now ETF-initialized and kept on the hypersphere."""\n    def __init__(self, n_classes=N_CLASSES, dim=SHARED_DIM, margin=0.5, init_prototypes=None):\n        super().__init__()\n        if init_prototypes is None:\n            init_prototypes = _init_proto_bank(n_classes=n_classes, dim=dim, use_etf=True)\n        init = torch.stack([init_prototypes[c] for c in range(n_classes)], dim=0)\n        self.prototypes = nn.Parameter(F.normalize(init, dim=-1))\n        self.margin = margin\n\n    def contrastive_update_loss(self, client_protos):\n        losses = []\n        g = F.normalize(self.prototypes, dim=-1)\n        for cp in client_protos:\n            for c, (local_proto, _n) in cp.items():\n                local_proto = F.normalize(local_proto, dim=-1)\n                pos = F.cosine_similarity(local_proto.unsqueeze(0), g[c].unsqueeze(0))\n                negs = torch.stack([F.cosine_similarity(local_proto.unsqueeze(0), g[k].unsqueeze(0))\n                                    for k in range(g.shape[0]) if k != c])\n                losses.append(F.relu(self.margin - (pos - negs.mean())))\n        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=g.device, requires_grad=True)\n\n    def as_dict(self):\n        g = F.normalize(self.prototypes, dim=-1).detach()\n        return {c: g[c] for c in range(g.shape[0])}\n\ndef fedproto_local_loss(emb, y, global_protos, temperature=None, class_weights=None):\n    """Prototypical contrastive loss: pull toward own-class global prototype, push from others."""\n    temperature = PROTO_TEMPERATURE if temperature is None else temperature\n    if not global_protos:\n        return torch.tensor(0.0, device=emb.device, requires_grad=True)\n\n    classes_present = sorted(global_protos.keys())\n    if len(classes_present) < 2:\n        return torch.tensor(0.0, device=emb.device, requires_grad=True)\n\n    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])\n    losses = []\n    for i in range(emb.shape[0]):\n        yi = int(y[i].item())\n        if yi not in global_protos:\n            continue\n        sims = F.cosine_similarity(F.normalize(emb[i].unsqueeze(0), dim=-1), proto_mat) / temperature\n        target = classes_present.index(yi)\n        ce = F.cross_entropy(sims.unsqueeze(0), torch.tensor([target], device=emb.device))\n        if class_weights is not None:\n            ce = ce * class_weights[yi]\n        losses.append(ce)\n    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=emb.device, requires_grad=True)\n\ndef intra_class_compactness_loss(emb, y, temperature=0.07):\n    emb = F.normalize(emb, dim=-1)\n    sim = emb @ emb.T\n    losses = []\n    for i in range(len(y)):\n        pos_mask = (y == y[i])\n        pos_mask[i] = False\n        neg_mask = (y != y[i])\n        if pos_mask.sum() == 0:\n            continue\n        pos = sim[i][pos_mask] / temperature\n        neg = sim[i][neg_mask] / temperature\n        numerator = torch.exp(pos).sum()\n        denominator = numerator + torch.exp(neg).sum()\n        losses.append(-torch.log(numerator / denominator))\n    if len(losses) == 0:\n        return torch.tensor(0.0, device=emb.device, requires_grad=True)\n    return torch.stack(losses).mean()\n\ndef classification_loss(logits, y, class_weights=None):\n    """Class-balanced classifier loss. Focal (default) emphasizes hard / minority examples,\n    which is what macro-F1 rewards on these imbalanced silos; falls back to weighted CE with\n    label smoothing when USE_FOCAL_LOSS is False."""\n    if USE_FOCAL_LOSS:\n        logp = F.log_softmax(logits, dim=-1)\n        ce = F.nll_loss(logp, y, weight=class_weights, reduction=\'none\')\n        pt = logp.gather(1, y.unsqueeze(1)).squeeze(1).exp()\n        return (((1.0 - pt) ** FOCAL_GAMMA) * ce).mean()\n    return F.cross_entropy(logits, y, weight=class_weights, label_smoothing=CE_LABEL_SMOOTHING)\n\ndef _mixup_ce(head, xb, yb, class_weights=None):\n    """Mixup on the classification path only (proto/compact losses keep hard labels)."""\n    if (not USE_MIXUP) or MIXUP_ALPHA <= 0 or xb.shape[0] < 2:\n        return classification_loss(head.classify(head(xb)), yb, class_weights)\n    lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))\n    perm = torch.randperm(xb.shape[0], device=xb.device)\n    xmix = lam * xb + (1.0 - lam) * xb[perm]\n    logits = head.classify(head(xmix))\n    return (lam * classification_loss(logits, yb, class_weights)\n            + (1.0 - lam) * classification_loss(logits, yb[perm], class_weights))\n\ndef positive_prob(head, emb):\n    """P(class=1) from the classifier head on a projected (normalized) embedding."""\n    return torch.softmax(head.classify(emb), dim=-1)[:, 1]\n\ndef best_threshold_on_val(head, Xval, yval):\n    """Binary only: choose the decision threshold that maximizes macro-F1 on this silo\'s VAL\n    set. Returns 0.5 for the multi-class task or when tuning is disabled."""\n    if N_CLASSES != 2 or not USE_THRESHOLD_TUNING \\\n       or PREDICT_WITH != \'classifier\' or getattr(head, \'classifier\', None) is None:\n        return 0.5\n    head.eval()\n    with torch.no_grad():\n        p = positive_prob(head, head(Xval)).cpu().numpy()\n    yv = yval.cpu().numpy() if torch.is_tensor(yval) else np.asarray(yval)\n    best_t, best_f1 = 0.5, -1.0\n    for t in np.linspace(0.05, 0.95, THRESHOLD_GRID):\n        f1 = f1_score(yv, (p >= t).astype(int), average=\'macro\', zero_division=0)\n        if f1 > best_f1:\n            best_f1, best_t = f1, float(t)\n    return best_t\n\ndef predict_labels(head, emb, global_protos, pos_threshold=None):\n    """Unified prediction: trained classifier head if enabled, else nearest-prototype.\n    For binary with a tuned pos_threshold, decide by P(class=1) >= threshold."""\n    if PREDICT_WITH == \'classifier\' and getattr(head, \'classifier\', None) is not None:\n        if N_CLASSES == 2 and pos_threshold is not None:\n            return (positive_prob(head, emb) >= pos_threshold).long()\n        return head.classify(emb).argmax(dim=-1)\n    return predict_via_prototypes(emb, global_protos)\n\ndef predict_via_prototypes(emb, global_protos):\n    classes_present = sorted(global_protos.keys())\n    if not classes_present:\n        return torch.zeros(len(emb), dtype=torch.long, device=emb.device)\n    proto_mat = torch.stack([F.normalize(global_protos[c], dim=-1) for c in classes_present])\n    sims = F.normalize(emb, dim=-1) @ F.normalize(proto_mat, dim=-1).T\n    idx = sims.argmax(dim=-1)\n    return torch.tensor([classes_present[i] for i in idx.tolist()], device=emb.device)\n\ndef per_class_reliability(y_true, y_pred, n_classes=N_CLASSES):\n    scores = f1_score(\n        y_true,\n        y_pred,\n        average=None,\n        labels=list(range(n_classes)),\n        zero_division=0\n    )\n    return {c: float(scores[c]) for c in range(n_classes)}\n\ndef class_weights_from_labels(y, n_classes=N_CLASSES, eps=1e-6):\n    """Inverse-frequency class weights (present classes only; absent -> 0).\n    Normalized so the mean weight over present classes is ~1."""\n    counts = torch.bincount(y, minlength=n_classes).float()\n    present = counts > 0\n    w = torch.zeros(n_classes)\n    w[present] = 1.0 / counts[present]\n    w[present] = w[present] * (present.sum().float() / w[present].sum().clamp_min(eps))\n    return w\n\ndef run_one_round(\n    tdata,\n    heads,\n    global_protos,\n    optimizers,\n    local_epochs=None,\n    trainable_global=None,\n    client_gates=None,\n    warmup=False,\n    use_reliability_weighting=None,\n    grad_clip=None\n):\n    # live-resolve so a search loop can vary these via globals\n    local_epochs = LOCAL_EPOCHS if local_epochs is None else local_epochs\n    use_reliability_weighting = USE_CLIENT_RELIABILITY_WEIGHTING if use_reliability_weighting is None else use_reliability_weighting\n    grad_clip = GRAD_CLIP if grad_clip is None else grad_clip\n\n    client_local_protos = []\n    client_reliability = {}\n    round_losses = []\n\n    for silo in tdata:\n\n        Xtr, ytr = tdata[silo]["train"]\n        Xva, yva = tdata[silo]["val"]\n\n        head = heads[silo]\n        opt = optimizers[silo]\n\n        gate = (\n            client_gates.get(silo, 0.5)\n            if client_gates is not None\n            else 0.5\n        )\n\n        head.train()\n        cw = class_weights_from_labels(ytr) if USE_CLASS_WEIGHTING else None\n        n_train = Xtr.shape[0]\n\n        for _ in range(local_epochs):\n\n            perm = torch.randperm(n_train)\n\n            for start in range(0, n_train, BATCH_SIZE):\n\n                idx = perm[start:start + BATCH_SIZE]\n                xb = Xtr[idx]\n                yb = ytr[idx]\n\n                # cheap augmentation: jitter embeddings to expand effective sample size\n                if AUGMENT_NOISE_STD > 0:\n                    xb = xb + AUGMENT_NOISE_STD * torch.randn_like(xb)\n\n                opt.zero_grad()\n\n                emb = head(xb)\n\n                proto_loss = (\n                    fedproto_local_loss(emb, yb, global_protos, class_weights=cw)\n                    if len(global_protos) > 0\n                    else torch.tensor(0.0, device=emb.device, requires_grad=True)\n                )\n\n                if USE_COMPACTNESS:\n                    compact_loss = intra_class_compactness_loss(emb, yb)\n                else:\n                    compact_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)\n\n                # FedPer-style supervised classification term (the headline predictor).\n                if USE_CLASSIFIER_HEAD and head.classifier is not None:\n                    ce_loss = _mixup_ce(head, xb, yb, class_weights=cw)\n                else:\n                    ce_loss = torch.tensor(0.0, device=emb.device, requires_grad=True)\n\n                # Per-silo personalization comes from `gate` alone (see config GATE_INIT).\n                total_loss = gate * proto_loss + COMPACT_LAMBDA * compact_loss + CE_WEIGHT * ce_loss\n\n                total_loss.backward()\n\n                torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)\n\n                opt.step()\n\n        head.eval()\n\n        with torch.no_grad():\n\n            emb_train = head(Xtr)\n\n            local_proto = compute_local_prototypes(\n                emb_train,\n                ytr\n            )\n\n            client_local_protos.append(\n                local_proto\n            )\n\n            emb_val = head(Xva)\n\n            pred_val = predict_via_prototypes(\n                emb_val,\n                global_protos\n            )\n\n            client_reliability[silo] = (\n                per_class_reliability(\n                    yva.cpu().numpy(),\n                    pred_val.cpu().numpy(),\n                    N_CLASSES\n                )\n            )\n\n        round_losses.append(\n            float(total_loss.item())\n        )\n\n    # -----------------------------------------\n    # Global prototype update\n    # -----------------------------------------\n    if trainable_global is not None:\n\n        proto_optimizer = torch.optim.Adam(\n            trainable_global.parameters(),\n            lr=1e-3\n        )\n\n        proto_optimizer.zero_grad()\n\n        gp_loss = (\n            trainable_global\n            .contrastive_update_loss(\n                client_local_protos\n            )\n        )\n\n        gp_loss.backward()\n\n        proto_optimizer.step()\n\n        with torch.no_grad():\n            trainable_global.prototypes.data = (\n                F.normalize(\n                    trainable_global.prototypes.data,\n                    dim=-1\n                )\n            )\n\n        new_global = (\n            trainable_global.as_dict()\n        )\n\n    else:\n\n        if (\n            use_reliability_weighting\n            and len(client_reliability) > 0\n        ):\n\n            new_global = (\n                aggregate_prototypes_reliability(\n                    client_local_protos,\n                    client_reliability\n                )\n            )\n\n        else:\n\n            new_global = (\n                aggregate_prototypes_naive(\n                    client_local_protos\n                )\n            )\n\n    return (\n        new_global,\n        client_local_protos,\n        float(np.mean(round_losses)),\n        client_reliability\n    )\n\nprint("Core FL functions defined (ETF init + warm-up + reliability-weighted aggregation enabled).")'

LOAD_SRC = '# ============================ Load labels + embeddings per silo (CONFIRMED sources) ============================\nlabels = pd.read_csv(LABELS_CSV)\nlabels["subject_id"] = labels["subject_id"].astype(str)\n\ndef load_eeg():\n    """Concatenate the three locked-split files; ids are MODMA subject_id strings e.g. \'2010001\'."""\n    Xs, ids, ys = [], [], []\n    for split in ("train", "val", "test"):\n        Xs.append(np.load(f"{EEG_EMBED_DIR}/eeg_embeddings_{split}.npy"))\n        ids.append(np.load(f"{EEG_EMBED_DIR}/eeg_subj_ids_{split}.npy", allow_pickle=True))\n    X = np.concatenate(Xs, axis=0)\n    ids = np.concatenate(ids, axis=0)\n    return X, ids\n\ndef load_text():\n    d = np.load(TEXT_EMBED_NPZ, allow_pickle=True)\n    return d["X"], d["ids"]\n\ndef load_audio():\n    """Reconstruct the LOCKED 1536-dim (layer-10 mean+std) input from raw per-participant turn\n    embeddings. Verified numerically identical to the original DAIC notebook\'s pool_mean_std()\n    + layer-slice -- this is NOT an approximation, it reproduces the exact locked-model input."""\n    ids, X = [], []\n    for f in sorted(glob.glob(f"{AUDIO_TURN_DIR}/*.npz")):\n        pid = int(os.path.basename(f).split(".")[0])\n        d = np.load(f)\n        emb = d["emb"]                                  # (n_turns, 13, 768)\n        layer_emb = emb[:, AUDIO_LOCK_LAYER, :]          # (n_turns, 768)\n        mu = layer_emb.mean(axis=0)\n        sd = layer_emb.std(axis=0)\n        X.append(np.concatenate([mu, sd]))               # (1536,)\n        ids.append(pid)\n    return np.stack(X).astype(np.float32), np.array(ids)\n\nprint("Loading EEG (confirmed 128-dim, n=55), TEXT (confirmed 768-dim, n=99),")\nprint("AUDIO (reconstructed 1536-dim from turn embeddings, n~189; this matches the LOCKED SVM input)...\\n")\ntry:\n    X_eeg_raw, ids_eeg = load_eeg();  print("EEG  :", X_eeg_raw.shape, "| dtype ids:", ids_eeg.dtype)\nexcept Exception as e: print("EEG load failed:", e)\ntry:\n    X_txt_raw, ids_txt = load_text(); print("Text :", X_txt_raw.shape, "| dtype ids:", ids_txt.dtype)\nexcept Exception as e: print("Text load failed:", e)\ntry:\n    X_aud_raw, ids_aud = load_audio(); print("Audio:", X_aud_raw.shape, "| dtype ids:", ids_aud.dtype,\n                                              "\\n   (NOTE: this is RECONSTRUCTED from turn_embeddings/*.npz,")\n    print("    NOT loaded from audio_embeddings_*.npy or X_audio.npy -- those are different, non-locked tracks.)")\nexcept Exception as e: print("Audio load failed:", e)'

SPLITS_SRC = '# ============================ Join to harmonized labels, build subject-level splits ============================\ndef join_silo(silo_name, X_raw, ids_raw):\n    """Join an embedding array to harmonized labels by subject_id, with a loud diagnostic if the\n    join rate is suspiciously low (catches silent id-format mismatches, e.g. zero-padding or\n    str-vs-int, before they corrupt the silo\'s training set)."""\n    sub = labels[labels.silo == silo_name].copy()\n    sub["subject_id"] = sub["subject_id"].astype(str)\n    id_to_row = {str(i): r for r, i in enumerate(ids_raw)}\n    keep_rows, keep_y = [], []\n    for _, row in sub.iterrows():\n        sid = row.subject_id\n        if sid in id_to_row and not pd.isna(row.y_unified):\n            keep_rows.append(id_to_row[sid]); keep_y.append(int(row.y_unified))\n    X = X_raw[keep_rows]\n    y = np.array(keep_y, dtype=np.int64)\n    if TASK == "binary":\n        _cut = BINARY_POSITIVE_FROM[silo_name] if isinstance(BINARY_POSITIVE_FROM, dict) else BINARY_POSITIVE_FROM\n        y = (y >= _cut).astype(np.int64)   # per-silo positive cut\n    join_rate = len(y) / max(len(sub), 1)\n    print(f"  [{silo_name}] joined {len(y)}/{len(sub)} subjects "\n          f"(dropped = no matching embedding id or missing label)")\n    if join_rate < 0.9:\n        print(f"  *** WARNING: only {join_rate:.0%} of {silo_name} subjects joined. ***")\n        print(f"      sample harmonized ids : {list(sub.subject_id.head(3))}")\n        print(f"      sample embedding ids  : {[str(i) for i in ids_raw[:3]]}")\n        print(f"      -> check id format (zero-padding, str-vs-int, leading characters) before trusting results.")\n    return X, y\n\nX_eeg, y_eeg = join_silo("eeg",   X_eeg_raw, ids_eeg)\nX_txt, y_txt = join_silo("text",  X_txt_raw, ids_txt)\nX_aud, y_aud = join_silo("audio", X_aud_raw, ids_aud)\n\nfrom sklearn.model_selection import train_test_split\ndef subject_split(X, y, val_frac=0.2, test_frac=0.2, seed=42):\n    """Subject-level (no-leakage) train/val/test split, stratified where class counts allow."""\n    n = len(y)\n    idx = np.arange(n)\n    try:\n        idx_tr, idx_rest = train_test_split(idx, test_size=val_frac+test_frac, random_state=seed, stratify=y)\n        rel_test = test_frac/(val_frac+test_frac)\n        idx_val, idx_te = train_test_split(idx_rest, test_size=rel_test, random_state=seed, stratify=y[idx_rest])\n    except ValueError:\n        # a class too small to stratify -- fall back to unstratified split, warn loudly\n        print(f"    WARNING: a class has too few subjects to stratify -- using unstratified split")\n        idx_tr, idx_rest = train_test_split(idx, test_size=val_frac+test_frac, random_state=seed)\n        rel_test = test_frac/(val_frac+test_frac)\n        idx_val, idx_te = train_test_split(idx_rest, test_size=rel_test, random_state=seed)\n    return (X[idx_tr], y[idx_tr]), (X[idx_val], y[idx_val]), (X[idx_te], y[idx_te])\n\nsplits = {\n    "eeg":   subject_split(X_eeg, y_eeg),\n    "text":  subject_split(X_txt, y_txt),\n    "audio": subject_split(X_aud, y_aud),\n}\nfor silo, (tr, va, te) in splits.items():\n    if TASK == "binary":\n        cut = BINARY_POSITIVE_FROM[silo] if isinstance(BINARY_POSITIVE_FROM, dict) else BINARY_POSITIVE_FROM\n        cut_str = f" | binary cut: y>={cut}"\n    else:\n        cut_str = ""\n    print(f"{silo:6s} train={len(tr[1])} val={len(va[1])} test={len(te[1])} "\n          f"| train class counts={np.bincount(tr[1], minlength=N_CLASSES).tolist()}" + cut_str)'

STANDALONE_SRC = 'from sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import f1_score\n\ndef fit_probe(X_tr, y_tr, seed):\n    sc = StandardScaler().fit(X_tr)\n    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed).fit(sc.transform(X_tr), y_tr)\n    return sc, clf\n\ndef eval_probe(sc, clf, X, y):\n    pred = clf.predict(sc.transform(X))\n    return f1_score(y, pred, average="macro", zero_division=0), pred\n\nstandalone_results = {}\nfor silo, (tr, va, te) in splits.items():\n    f1s = []\n    for seed in range(N_SEEDS):\n        sc, clf = fit_probe(*tr, seed=seed)\n        f1, _ = eval_probe(sc, clf, *te)\n        f1s.append(f1)\n    standalone_results[silo] = (np.mean(f1s), np.std(f1s))\n    print(f"[standalone] {silo:6s} test macro-F1: {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}  (n_seeds={N_SEEDS})")'

TDATA_SRC = 'def to_tensor_data(splits, standardize=USE_INPUT_STANDARDIZATION):\n    """Per-silo z-scoring of head inputs using TRAIN stats only (no leakage).\n    The standalone LR baseline already standardizes its inputs; the FL heads did not,\n    which handicapped them on raw-scale embeddings. This puts both on equal footing."""\n    out = {}\n    for silo, (tr, va, te) in splits.items():\n        Xtr, ytr = tr; Xva, yva = va; Xte, yte = te\n        if standardize:\n            mu = Xtr.mean(axis=0, keepdims=True)\n            sd = Xtr.std(axis=0, keepdims=True) + 1e-6\n            Xtr = (Xtr - mu) / sd; Xva = (Xva - mu) / sd; Xte = (Xte - mu) / sd\n        out[silo] = {\n            "train": (torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long)),\n            "val":   (torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva, dtype=torch.long)),\n            "test":  (torch.tensor(Xte, dtype=torch.float32), torch.tensor(yte, dtype=torch.long)),\n        }\n    return out\n\ntdata = to_tensor_data(splits)\nin_dims = {silo: tdata[silo]["train"][0].shape[1] for silo in tdata}\nprint("input dims per client:", in_dims)\n\ndef per_silo_val_macro_f1(heads, global_protos, tdata):\n    """Per-silo VAL macro-F1 dict (no leakage; never touches test). Uses the unified\n    predictor (trained classifier head if enabled, else nearest-prototype)."""\n    out = {}\n    for silo in tdata:\n        heads[silo].eval()\n        with torch.no_grad():\n            emb = heads[silo](tdata[silo]["val"][0])\n            pred = predict_labels(heads[silo], emb, global_protos)\n        out[silo] = f1_score(tdata[silo]["val"][1].numpy(), pred.cpu().numpy(),\n                             average="macro", zero_division=0)\n    return out\n\ndef pooled_val_macro_f1(heads, global_protos, tdata):\n    """Mean per-silo VAL macro-F1 (used for early stopping)."""\n    d = per_silo_val_macro_f1(heads, global_protos, tdata)\n    return float(np.mean(list(d.values())))\n\ndef run_fl_training(\n    tdata,\n    in_dims,\n    seed,\n    use_trainable_global=False,\n    n_rounds=N_ROUNDS,\n    patience=EARLY_STOP_PATIENCE,\n    lr=None,\n    use_sample_size_weighting=USE_SAMPLE_SIZE_WEIGHTING\n):\n\n    torch.manual_seed(seed)\n\n    lr = lr if lr is not None else LR_HEAD\n\n    heads = {\n        silo: ProjectionHead(in_dims[silo])\n        for silo in tdata\n    }\n\n    optimizers = {\n        silo: torch.optim.AdamW(\n            h.parameters(),\n            lr=lr,\n            weight_decay=1e-4\n        )\n        for silo, h in heads.items()\n    }\n\n    trainable_global = (\n        TrainableGlobalPrototypes(\n            n_classes=N_CLASSES,\n            dim=SHARED_DIM,\n            init_prototypes=_init_proto_bank(\n                N_CLASSES,\n                SHARED_DIM,\n                use_etf=USE_ETF_INIT\n            )\n        )\n        if use_trainable_global\n        else None\n    )\n\n    global_protos = (\n        trainable_global.as_dict()\n        if trainable_global is not None\n        else _init_proto_bank(\n            N_CLASSES,\n            SHARED_DIM,\n            use_etf=USE_ETF_INIT\n        )\n    )\n\n    client_data = {\n        silo: tdata[silo]["train"]\n        for silo in tdata\n    }\n\n    loss_weights = (\n        compute_sample_size_weights(client_data)\n        if use_sample_size_weighting\n        else None\n    )\n\n    # -----------------------------\n    # Personalized initial gates -- now reads GATE_INIT directly (single source of truth).\n    # The previous version duplicated these as a separate hardcoded literal here, so editing\n    # GATE_INIT in the config cell silently had NO effect on actual runs -- a latent bug.\n    # -----------------------------\n    client_gates = dict(GATE_INIT)\n\n    best_val = -1.0\n    best_round = -1\n    best_state = None\n\n    best_protos = {\n        c: v.clone()\n        for c, v in global_protos.items()\n    }\n\n    no_improve = 0\n    history = []\n\n    for r in range(n_rounds):\n\n        warmup = (r < WARMUP_ROUNDS)\n\n        global_protos, _, loss, client_reliability = run_one_round(\n            tdata=tdata,\n            heads=heads,\n            global_protos=global_protos,\n            optimizers=optimizers,\n            trainable_global=trainable_global,\n            client_gates=client_gates,\n            warmup=warmup,\n            use_reliability_weighting=USE_CLIENT_RELIABILITY_WEIGHTING\n        )\n\n        # --------------------------------------\n        # Improved gate adaptation\n        # --------------------------------------\n        if USE_CLIENT_RELIABILITY_WEIGHTING:\n\n            for silo in client_gates:\n\n                rel_vals = list(\n                    client_reliability.get(\n                        silo,\n                        {}\n                    ).values()\n                )\n\n                rel = (\n                    float(np.mean(rel_vals))\n                    if len(rel_vals) > 0\n                    else 0.5\n                )\n\n                rel = float(\n                    np.clip(\n                        rel,\n                        GATE_MIN,\n                        GATE_MAX\n                    )\n                )\n\n                # Audio stays local -- RECALIBRATED (see GATE_INIT comment above): bakes the\n                # empirically-known-good effective strength directly into this single formula\n                # instead of relying on a second hardcoded multiplier elsewhere in the loss.\n                if silo == "audio":\n\n                    target_gate = min(\n                        rel * 0.01,\n                        0.0075\n                    )\n\n                # Text moderately local\n                elif silo == "text":\n\n                    target_gate = min(\n                        rel * 0.60,\n                        0.50\n                    )\n\n                # EEG gets most global guidance\n                else:\n\n                    target_gate = rel\n\n                client_gates[silo] = (\n                    GATE_ADAPT_MOMENTUM\n                    * client_gates[silo]\n                    +\n                    (1.0 - GATE_ADAPT_MOMENTUM)\n                    * target_gate\n                )\n\n        val_f1 = pooled_val_macro_f1(\n            heads,\n            global_protos,\n            tdata\n        )\n\n        history.append({\n            "round": r,\n            "loss": loss,\n            "val_macro_f1": val_f1,\n            "warmup": warmup,\n            **{\n                f"gate_{s}": float(client_gates[s])\n                for s in client_gates\n            }\n        })\n\n        if val_f1 > best_val:\n\n            best_val = val_f1\n            best_round = r\n\n            best_state = {\n                silo: {\n                    k: v.clone()\n                    for k, v in h.state_dict().items()\n                }\n                for silo, h in heads.items()\n            }\n\n            best_protos = {\n                c: v.clone()\n                for c, v in global_protos.items()\n            }\n\n            no_improve = 0\n\n        else:\n            no_improve += 1\n\n        if no_improve >= patience:\n            break\n\n    for silo, h in heads.items():\n        h.load_state_dict(\n            best_state[silo]\n        )\n\n    return (\n        heads,\n        best_protos,\n        best_round,\n        pd.DataFrame(history)\n    )\nprint("FL training loop defined: ETF init, warm-up, adaptive gates, and early stopping on pooled val macro-F1.")\n'

print('core source embedded:',
      {k: len(v) for k, v in [('config', CFG_SRC), ('classes', CLASSES_SRC), ('load', LOAD_SRC),
                              ('splits', SPLITS_SRC), ('standalone', STANDALONE_SRC),
                              ('tdata+run_fl', TDATA_SRC)]})

## 5. `run_setup(task)`

In [ ]:
def _wrap_run_fl_training():
    fn = globals()["run_fl_training"]
    if getattr(fn, "_seeded_wrapper", False):
        return
    orig = fn
    @functools.wraps(orig)
    def wrapped(tdata, in_dims, seed, *a, **kw):
        set_all_seeds(seed)
        return orig(tdata, in_dims, seed, *a, **kw)
    wrapped._seeded_wrapper = True
    globals()["run_fl_training"] = wrapped


def run_setup(task: str, verbose: bool = False):
    assert task in ("binary", "severity3")
    globals()["_TASK"] = task
    import io, contextlib
    buf = io.StringIO()
    ctx = contextlib.nullcontext() if verbose else contextlib.redirect_stdout(buf)
    with ctx:
        for src in (CFG_SRC, CLASSES_SRC, LOAD_SRC, SPLITS_SRC, STANDALONE_SRC, TDATA_SRC):
            exec(compile(src, f"<{task}>", "exec"), globals())
    _wrap_run_fl_training()
    print(f"[setup] task={task} | N_CLASSES={N_CLASSES} | in_dims={in_dims}")
    print(f"[setup] test sizes: " + ", ".join(f"{s}={len(tdata[s]['test'][1])}" for s in tdata))
    print(f"[setup] standalone: " + ", ".join(f"{s}={standalone_results[s][0]:.3f}" for s in tdata))
    return tdata, in_dims


def determinism_selftest(seed=0):
    h1, _p1, r1, _ = run_fl_training(tdata, in_dims, seed, use_trainable_global=False, lr=LR_HEAD)
    h2, _p2, r2, _ = run_fl_training(tdata, in_dims, seed, use_trainable_global=False, lr=LR_HEAD)
    md_ = 0.0
    for s in tdata:
        h1[s].eval(); h2[s].eval()
        with torch.no_grad():
            md_ = max(md_, float((h1[s](tdata[s]["test"][0]) - h2[s](tdata[s]["test"][0])).abs().max()))
    ok = (r1 == r2) and md_ < 1e-6
    print(f"  locked round: {r1} vs {r2} | max|dz| = {md_:.3e} -> {'MATCH (deterministic)' if ok else 'MISMATCH'}")
    if not ok:
        print("  !! Still nondeterministic. Ensure CUBLAS_WORKSPACE_CONFIG was set before torch import,")
        print("     then Runtime > Restart session and re-run from the top.")
    return ok

## 6. Shared evaluation helpers

In [ ]:
def eval_config(use_trainable_global=None, local_only=False, n_seeds=None):
    # Run all seeds for one configuration.
    n_seeds = n_seeds or N_SEEDS
    per_seed = {s: [] for s in tdata}
    unw, sw = [], []
    for seed in range(n_seeds):
        if local_only:
            snap = globals().get("GATE_INIT")
            globals()["GATE_INIT"] = {k: 0.0 for k in tdata}     # no global-prototype pull
            heads, protos, _, _ = run_fl_training(tdata, in_dims, seed,
                                                  use_trainable_global=False, lr=LR_HEAD)
            globals()["GATE_INIT"] = snap
        else:
            heads, protos, _, _ = run_fl_training(tdata, in_dims, seed,
                                                  use_trainable_global=use_trainable_global,
                                                  lr=LR_HEAD)
        yt_all, yp_all = [], []
        for s in tdata:
            heads[s].eval()
            thr = (best_threshold_on_val(heads[s], tdata[s]["val"][0], tdata[s]["val"][1])
                   if N_CLASSES == 2 else None)
            with torch.no_grad():
                emb = heads[s](tdata[s]["test"][0])
            pred = (predict_labels(heads[s], emb, protos, pos_threshold=thr) if N_CLASSES == 2
                    else predict_labels(heads[s], emb, protos))
            yt = tdata[s]["test"][1].numpy(); yp = pred.cpu().numpy()
            per_seed[s].append(f1_score(yt, yp, average="macro", zero_division=0))
            yt_all.append(yt); yp_all.append(yp)
        unw.append(np.mean([per_seed[s][-1] for s in tdata]))
        sw.append(f1_score(np.concatenate(yt_all), np.concatenate(yp_all),
                           average="macro", zero_division=0))
    mean = {s: float(np.mean(v)) for s, v in per_seed.items()}
    std  = {s: float(np.std(v))  for s, v in per_seed.items()}
    return mean, std, per_seed, float(np.mean(unw)), float(np.mean(sw))


def collect_predictions(use_trainable_global, n_seeds=None):
    n_seeds = n_seeds or N_SEEDS
    out = {s: {"y": [], "p": []} for s in tdata}
    for seed in range(n_seeds):
        heads, protos, _, _ = run_fl_training(tdata, in_dims, seed,
                                              use_trainable_global=use_trainable_global, lr=LR_HEAD)
        for s in tdata:
            heads[s].eval()
            with torch.no_grad():
                emb = heads[s](tdata[s]["test"][0])
            pred = predict_labels(heads[s], emb, protos)
            out[s]["y"].append(tdata[s]["test"][1].numpy())
            out[s]["p"].append(pred.cpu().numpy())
    for s in out:
        out[s]["y"] = np.concatenate(out[s]["y"])
        out[s]["p"] = np.concatenate(out[s]["p"])
    return out


def to_latex(df, caption, label, floatfmt="%.3f"):
    return df.to_latex(index=False, escape=False, caption=caption, label=label,
                       float_format=lambda x: floatfmt % x)

RESULTS = {}   # everything lands here

## 7. Binary task — Tables 2 & 3

In [ ]:
tdata, in_dims = run_setup("binary")

print("\n[determinism self-test]")
assert determinism_selftest(0), "Nondeterministic — fix before trusting any numbers."

print("\nRunning binary configurations...")
bin_res = {}
bin_res["standalone"] = ({s: standalone_results[s][0] for s in tdata},
                         {s: standalone_results[s][1] for s in tdata}, None, None, None)
bin_res["local"]    = eval_config(local_only=True)
bin_res["fedproto"] = eval_config(use_trainable_global=False)
bin_res["fedtgp"]   = eval_config(use_trainable_global=True)

rows = []
for s in tdata:
    rows.append({
        "Silo": s.upper(),
        "Standalone":      bin_res["standalone"][0][s],
        "Local classifier": bin_res["local"][0][s],
        "Vanilla FedProto": bin_res["fedproto"][0][s],
        "FedTGP-style":     bin_res["fedtgp"][0][s],
    })
binary_df = pd.DataFrame(rows)
print("\n=== BINARY: per-silo macro-F1 (mean over seeds) ===")
print(binary_df.round(3).to_string(index=False))

overall_df = pd.DataFrame([
    {"Method": "Standalone",             "Overall (unweighted)": np.mean(list(bin_res['standalone'][0].values())), "Overall (sample-weighted)": np.nan},
    {"Method": "Local classifier (no FL)","Overall (unweighted)": bin_res['local'][3],    "Overall (sample-weighted)": bin_res['local'][4]},
    {"Method": "Vanilla FedProto",        "Overall (unweighted)": bin_res['fedproto'][3], "Overall (sample-weighted)": bin_res['fedproto'][4]},
    {"Method": "FedTGP-style",            "Overall (unweighted)": bin_res['fedtgp'][3],   "Overall (sample-weighted)": bin_res['fedtgp'][4]},
])
print("\n=== BINARY: overall ===")
print(overall_df.round(3).to_string(index=False))

RESULTS["binary_per_silo"] = binary_df
RESULTS["binary_overall"]  = overall_df
RESULTS["binary_per_seed"] = {k: v[2] for k, v in bin_res.items() if v[2] is not None}
RESULTS["binary_std"]      = {k: v[1] for k, v in bin_res.items()}

binary_df.to_csv(f"{OUT}/final_binary_per_silo.csv", index=False)
overall_df.to_csv(f"{OUT}/final_binary_overall.csv", index=False)
print(f"\nsaved -> {OUT}/final_binary_per_silo.csv, final_binary_overall.csv")

## 8. Paired significance tests (binary)

In [ ]:
def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    sh = stats.shapiro(d).pvalue if len(np.unique(d)) > 1 else np.nan
    t, tp = stats.ttest_rel(a, b)
    if np.count_nonzero(d) == 0:
        w, wp = np.nan, np.nan
    else:
        w, wp = stats.wilcoxon(a, b)
    return float(d.mean()), float(sh), float(t), float(tp), float(wp)

ps = RESULTS["binary_per_seed"]
rows = []
for s in tdata:
    md_, sh, t, tp, wp = paired(ps["fedtgp"][s], ps["fedproto"][s])
    rows.append({"Comparison": "FedTGP-style vs. vanilla FedProto", "Silo": s.upper(),
                 "Mean diff": md_, "t": t, "p": tp, "Shapiro p": sh, "Wilcoxon p": wp})
for cfg, nm in [("fedproto", "Vanilla FedProto"), ("fedtgp", "FedTGP-style")]:
    for s in tdata:
        floor = standalone_results[s][0]
        v = np.asarray(ps[cfg][s], float)
        if np.std(v) == 0:
            continue                      # e.g. EEG identical on every seed
        t, p = stats.ttest_1samp(v, floor)
        rows.append({"Comparison": f"{nm} vs. standalone", "Silo": s.upper(),
                     "Mean diff": float(v.mean() - floor), "t": float(t), "p": float(p),
                     "Shapiro p": np.nan, "Wilcoxon p": np.nan})

stats_df = pd.DataFrame(rows)
print("=== Paired significance tests (binary) ===")
print(stats_df.round(4).to_string(index=False))
print(f"\nNote: with n={N_SEEDS} pairs the two-sided Wilcoxon test cannot reach p<0.05")
print("(minimum attainable p = 0.0625), so the paired t-test is primary. Tests are descriptive.")

RESULTS["stats"] = stats_df
stats_df.to_csv(f"{OUT}/final_stats.csv", index=False)
print(f"saved -> {OUT}/final_stats.csv")

## 9. 3-class severity — Table 4, per-class F1, confusion matrices

In [ ]:
tdata, in_dims = run_setup("severity3")

print("\nRunning 3-class configurations...")
sev = {}
sev["standalone"] = ({s: standalone_results[s][0] for s in tdata}, None, None, None, None)
sev["fedproto"]   = eval_config(use_trainable_global=False)
sev["fedtgp"]     = eval_config(use_trainable_global=True)

rows = []
for s in tdata:
    rows.append({"Silo": s.upper(),
                 "Standalone": sev["standalone"][0][s],
                 "FedProto":   sev["fedproto"][0][s],
                 "FedTGP":     sev["fedtgp"][0][s]})
sev_df = pd.DataFrame(rows)
sev_df.loc[len(sev_df)] = {"Silo": "Overall",
                           "Standalone": np.mean([sev['standalone'][0][s] for s in tdata]),
                           "FedProto":   sev['fedproto'][3],
                           "FedTGP":     sev['fedtgp'][3]}
print("\n=== 3-CLASS severity: macro-F1 ===")
print(sev_df.round(3).to_string(index=False))

RESULTS["severity3"] = sev_df
sev_df.to_csv(f"{OUT}/final_severity3.csv", index=False)
print(f"saved -> {OUT}/final_severity3.csv")

In [ ]:
CLASS_NAMES = ["none-minimal", "mild-moderate", "severe"]
preds = {"vanilla": collect_predictions(False), "fedtgp": collect_predictions(True)}

perclass = {}
for cfg, pd_ in preds.items():
    rows = []
    for s in tdata:
        y, p = pd_[s]["y"], pd_[s]["p"]
        f = f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
        rows.append({"Silo": s.upper(),
                     "None-minimal": f[0], "Mild-moderate": f[1], "Severe": f[2],
                     "Macro-F1": f1_score(y, p, average="macro", zero_division=0),
                     "Support": str(np.bincount(y, minlength=3).tolist())})
    perclass[cfg] = pd.DataFrame(rows)
    print(f"\n=== Per-class F1 (3-class, {cfg}) ===")
    print(perclass[cfg].round(3).to_string(index=False))
    perclass[cfg].to_csv(f"{OUT}/final_perclass_{cfg}.csv", index=False)

RESULTS["perclass"] = perclass

# confusion matrices
for cfg, pd_ in preds.items():
    silos = list(tdata)
    fig, axes = plt.subplots(1, len(silos), figsize=(4.6 * len(silos), 4.2))
    axes = np.atleast_1d(axes)
    for ax, s in zip(axes, silos):
        cm = confusion_matrix(pd_[s]["y"], pd_[s]["p"], labels=[0, 1, 2])
        ConfusionMatrixDisplay(cm, display_labels=["none-min", "mild-mod", "severe"]).plot(
            ax=ax, colorbar=False, cmap="Blues", values_format="d")
        ax.set_title(f"{s.upper()} ({cfg})")
        for lb in ax.get_xticklabels():
            lb.set_rotation(30); lb.set_ha("right")
    fig.suptitle(f"3-class confusion matrices ({cfg}, pooled over {N_SEEDS} seeds)", y=1.03)
    fig.tight_layout()
    fig.savefig(f"{OUT}/final_confusion_{cfg}.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"saved -> {OUT}/final_confusion_{cfg}.png")

## 10. Component ablation (binary)

In [ ]:
if RUN_ABLATION:
    tdata, in_dims = run_setup("binary")

    KNOBS = ["USE_ETF_INIT", "USE_FOCAL_LOSS", "USE_CLASSIFIER_HEAD", "PREDICT_WITH",
             "USE_COMPACTNESS", "USE_CLIENT_RELIABILITY_WEIGHTING",
             "GATE_INIT", "GATE_ADAPT_MOMENTUM"]

    def snap():    return {k: copy.deepcopy(globals()[k]) for k in KNOBS if k in globals()}
    def restore(s):
        for k, v in s.items(): globals()[k] = copy.deepcopy(v)

    full_gate = dict(GATE_INIT)
    OVERRIDES = {
        "full (all components)":              {},
        "- ETF init":                         {"USE_ETF_INIT": False},
        "- adaptive gating (uniform g=1)":    {"GATE_INIT": {k: 1.0 for k in full_gate},
                                               "GATE_ADAPT_MOMENTUM": 1.0},
        "- gate adaptation (freeze at init)": {"GATE_ADAPT_MOMENTUM": 1.0},
        "- focal loss":                       {"USE_FOCAL_LOSS": False},
        "- classifier head":                  {"USE_CLASSIFIER_HEAD": False,
                                               "PREDICT_WITH": "prototype"},
        "- compactness":                      {"USE_COMPACTNESS": False},
        "- reliability weighting":            {"USE_CLIENT_RELIABILITY_WEIGHTING": False},
    }

    recs, base = [], snap()
    try:
        for variant, tgp in [("fedproto", False), ("fedtgp", True)]:
            print(f"\n===== {variant} =====")
            for name, ov in OVERRIDES.items():
                restore(base)
                for k, v in ov.items():
                    globals()[k] = v
                mean, std, _, unw, sw = eval_config(use_trainable_global=tgp)
                rec = {"variant": variant, "ablation": name,
                       **{s: mean[s] for s in tdata},
                       **{s + "_std": std[s] for s in tdata},
                       "overall_unweighted": unw, "overall_sample_weighted": sw}
                recs.append(rec)
                print(f"  {name:34s} " + " ".join(f"{s}={mean[s]:.3f}" for s in tdata)
                      + f" | unw={unw:.3f}")
    finally:
        restore(base)

    abl_df = pd.DataFrame(recs)
    RESULTS["ablation"] = abl_df
    abl_df.to_csv(f"{OUT}/final_ablation.csv", index=False)

    # deltas vs full
    drows = []
    for variant in abl_df.variant.unique():
        sub = abl_df[abl_df.variant == variant]
        full = sub[sub.ablation == "full (all components)"].iloc[0]
        for _, r in sub.iterrows():
            if r.ablation == "full (all components)":
                continue
            drows.append({"variant": variant, "removed": r.ablation.lstrip("- "),
                          **{s: round(float(full[s] - r[s]), 4) for s in tdata},
                          "overall": round(float(full.overall_unweighted - r.overall_unweighted), 4)})
    delta_df = pd.DataFrame(drows)
    print("\n=== Ablation deltas (drop when removed; positive = component helps) ===")
    print(delta_df.to_string(index=False))
    RESULTS["ablation_delta"] = delta_df
    delta_df.to_csv(f"{OUT}/final_ablation_deltas.csv", index=False)
    print(f"\nsaved -> {OUT}/final_ablation.csv, final_ablation_deltas.csv")
else:
    print("RUN_ABLATION = False -> skipped")

## 11. Shared space before vs after alignment

In [ ]:
if RUN_TSNE:
    tdata, in_dims = run_setup("binary")

    def project(heads):
        Z, Y, S = [], [], []
        for s in tdata:
            heads[s].eval()
            with torch.no_grad():
                Z.append(heads[s](tdata[s]["test"][0]).cpu().numpy())
            Y.append(tdata[s]["test"][1].numpy())
            S += [s] * len(Z[-1])

        return np.concatenate(Z), np.concatenate(Y), np.array(S)

    set_all_seeds(0)

    heads_b = {s: ProjectionHead(in_dims[s]) for s in tdata}
    protos_b = _init_proto_bank(N_CLASSES, SHARED_DIM, use_etf=USE_ETF_INIT)

    Zb, Yb, Sb = project(heads_b)

    heads_a, protos_a, _, _ = run_fl_training(
        tdata,
        in_dims,
        0,
        use_trainable_global=False,
        lr=LR_HEAD,
    )

    Za, Ya, Sa = project(heads_a)

    PB = np.stack([protos_b[c].cpu().numpy() for c in range(N_CLASSES)])
    PA = np.stack([protos_a[c].cpu().numpy() for c in range(N_CLASSES)])

    def reduce2d(Z, P, seed=0):
        X = np.vstack([Z, P])
        n = len(Z)

        try:
            import umap

            reducer = umap.UMAP(
                n_neighbors=min(15, n - 1),
                min_dist=0.1,
                metric="cosine",
                random_state=seed,
            )

        except Exception:
            from sklearn.manifold import TSNE

            reducer = TSNE(
                n_components=2,
                perplexity=min(15, max(5, n // 4)),
                init="pca",
                metric="cosine",
                random_state=seed,
            )

        E = reducer.fit_transform(X)
        return E[:n], E[n:]

    pb, prb = reduce2d(Zb, PB)
    pa, pra = reduce2d(Za, PA)

    colors = ["#1f77b4", "#d62728", "#2ca02c"][:N_CLASSES]
    markers = {s: m for s, m in zip(tdata.keys(), ["o", "s", "^"])}

    def panel(ax, pts, pr, Y, S, title):

        for silo in tdata:
            for cls in range(N_CLASSES):

                mask = (S == silo) & (Y == cls)

                if mask.any():
                    ax.scatter(
                        pts[mask, 0],
                        pts[mask, 1],
                        c=colors[cls],
                        marker=markers[silo],
                        s=42,
                        alpha=0.85,
                        edgecolors="k",
                        linewidths=0.3,
                    )

        # Global prototypes
        for cls in range(N_CLASSES):
            ax.scatter(
                pr[cls, 0],
                pr[cls, 1],
                c=colors[cls],
                marker="*",
                s=420,
                edgecolors="k",
                linewidths=1.0,
                zorder=5,
            )

        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    # ---------------- Plot ----------------
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))

    panel(ax[0], pb, prb, Yb, Sb,
          "Before alignment (untrained heads)")

    panel(ax[1], pa, pra, Ya, Sa,
          "After FL alignment")

    fig.suptitle(
        "Shared 512-d space: same class, different silos",
        y=0.98
    )

    # Reserve a little space for the title
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(
        f"{OUT}/final_shared_space.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(f"saved -> {OUT}/final_shared_space.png")

else:
    print("RUN_TSNE = False -> skipped")

## 12. LaTeX export

In [ ]:
print("%" * 70)
print("% BINARY -- per silo")
print(to_latex(RESULTS["binary_per_silo"],
               "Per-silo macro-F1, binary task.", "tab:binary_results"))
print("% BINARY -- overall")
print(to_latex(RESULTS["binary_overall"],
               "Overall macro-F1, binary task.", "tab:overall_binary"))
print("% 3-CLASS")
print(to_latex(RESULTS["severity3"],
               "Three-class severity macro-F1.", "tab:threeclass"))
for cfg, df in RESULTS.get("perclass", {}).items():
    print(f"% PER-CLASS ({cfg})")
    print(to_latex(df, f"Per-class F1, 3-class severity ({cfg}).", f"tab:perclass_{cfg}"))
print("% STATS")
print(to_latex(RESULTS["stats"].drop(columns=["Shapiro p", "Wilcoxon p"]),
               "Paired significance tests, binary task.", "tab:stats", floatfmt="%.3f"))
if "ablation_delta" in RESULTS:
    print("% ABLATION DELTAS")
    print(to_latex(RESULTS["ablation_delta"],
                   "Change in macro-F1 when each component is removed.", "tab:ablation"))

print("\n\nAll CSVs written to:", OUT)
print("Send these back:  final_binary_per_silo.csv, final_binary_overall.csv,")
print("                  final_severity3.csv, final_perclass_*.csv, final_stats.csv,")
print("                  final_ablation*.csv  (+ the two PNGs)")